# PROC NESTED による保険金精算のばらつきの組織階層別分解

## エグゼクティブサマリー

損害保険会社は、保険金精算における不整合が**どこから**生じているのかを知りたいと考えている。地域間の違いによるものか、支店間の違いによるものか、担当者ごとの違いによるものか、それとも単なる請求ごとのノイズによるものか。本ノートブックは、完全に入れ子構造を持つ自動車保険金請求の合成データ（地域 &gt; 支店 &gt; 担当者 &gt; 請求）を構築し、**PROC NESTED** を用いてランダム効果の分散分析を行い、階層の各レベルが寄与する分散成分を推定し、それぞれを総分散に対する割合として報告する。この結果は、経営陣が精算の一貫性を改善するためにどの組織階層に注力すべきかを示す。

## データソース

すべてのデータは最初の DATA ステップでインラインに生成される — 外部ファイルもネットワークも使用しない。設計は**完全な入れ子構造**である。各支店は必ず1つの地域に属し、各担当者は必ず1つの支店に属し、各請求は必ず1人の担当者に属する。

| データセット | 行数 | 変数 | 型 | 役割 | 説明 |
|---------|------|----------|------|------|-------------|
| `claims` | 40 | `Region` | 文字 | CLASS（レベル1、最も外側） | 地域（5地域: 関東、近畿、中部、東北、九州） |
| | | `Branch` | 文字 | CLASS（レベル2、Region に入れ子） | 支店コード（各地域あたり2支店） |
| | | `Adjuster` | 文字 | CLASS（レベル3、Branch に入れ子） | 損害査定担当者ID（各支店あたり2名） |
| | | `Settlement` | 数値 | VAR（従属変数） | 自動車保険金請求に対する支払保険金額（USD） |
| | | `CycleDays` | 数値 | VAR（従属変数） | 事故発生報告から保険金支払いまでの日数 |

合成構造: 5地域 x 2支店 x 2担当者 x 2件の請求 = 40 観測。地域のランダム効果、地域内の支店間ランダム効果、支店内の担当者間ランダム効果、そして請求レベルの残差が `rand('NORMAL', ...)` によって加法的に重ね合わされているため、分散成分を復元できる。地域効果は最も大きい標準偏差（2,200）で生成されており、請求が*どこで*処理されるかが支配的な要因となるよう設計されている。シードは `call streaminit(20260531)` で固定。コンパクトで完全にバランスの取れた設計により、階層のすべてのレベルに実質的な自由度が確保されている。

# PROC NESTED による保険金精算のばらつきの分解

損害保険会社が自動車保険金の支払額を見直すと、しばしば大きなばらつきが見つかる。その一部は避けられないもの（事故は一つひとつ異なる）だが、一部は**組織的な**要因を反映している — ある地域は他より支払いが手厚い、ある支店はより寛容である、特定の担当者が外れ値になっている、といった具合である。

このデータは厳密に**入れ子（階層）**構造を持つ。

```
地域  ->  支店（地域に入れ子）  ->  担当者（支店に入れ子）  ->  個々の請求
```

支店は必ず1つの地域に属し、担当者は必ず1つの支店に属するため、これらの要因は交差ではなく入れ子になっている。`PROC NESTED` はまさにこの設計に対してランダム効果の分散分析を行い、各レベルの**分散成分**を総分散に対する割合として推定する。この割合の内訳がビジネス上の問いに答える。*精算をより一貫させるために、組織のどの階層を対象とすべきか？*

## ステップ1 &mdash; 完全に入れ子構造を持つ合成の請求データを生成する

5地域、各地域に2支店、各支店に2担当者、各担当者が2件ずつ請求を処理する（合計40行）データをシミュレートする。各レベルが確実に分散へ寄与するよう、応答を加法的なランダム効果から構築する。

- **地域**効果（最も広がりが大きい — 地域間の差が最も大きい）、
- **地域内の支店**効果、
- **支店内の担当者**効果、
- そして**請求レベルの残差**（担当者内のノイズ）。

`call streaminit` は再現性のためにシードを固定し、各効果は `rand('NORMAL', mean, sd)` で生成される。地域効果は最も広い標準偏差を用いているため、総分散のうち最大の割合を占めるはずである。また、同じ階層構造を共有する2つ目の応答 `CycleDays` も生成し、後で複数応答の分析を行う。

In [1]:
データ claims;
   呼出 streaminit(20260531);
   長さ Region $8 Branch $12 Adjuster $16;

   /* 地域レベルのランダム効果: 地域間の差が最も大きい */
   繰返 r = 1 から 5;
      Region = scan('関東 近畿 中部 東北 九州', r);
      RegionEffect  = rand('NORMAL', 0, 2200);
      RegionCycle   = rand('NORMAL', 0, 11);

      /* 地域に入れ子になった支店 */
      繰返 b = 1 から 2;
         Branch = catx('-', Region, put(b, z2.));
         BranchEffect = rand('NORMAL', 0, 700);
         BranchCycle  = rand('NORMAL', 0, 4);

         /* 支店に入れ子になった担当者 */
         繰返 a = 1 から 2;
            Adjuster = catx('-', Branch, put(a, z1.));
            AdjEffect = rand('NORMAL', 0, 450);
            AdjCycle  = rand('NORMAL', 0, 2.5);

            /* この担当者が処理した個々の請求 */
            繰返 claim = 1 から 2;
               Settlement = 8500
                          + RegionEffect + BranchEffect + AdjEffect
                          + rand('NORMAL', 0, 1100);
               CycleDays  = 21
                          + RegionCycle + BranchCycle + AdjCycle
                          + rand('NORMAL', 0, 6);
               もし Settlement < 0 なら Settlement = 0;
               もし CycleDays  < 1 なら CycleDays  = 1;
               出力;
            終了;
         終了;
      終了;
   終了;

   保持 Region Branch Adjuster Settlement CycleDays;
実行;



NOTE: DATA claims


NOTE: Wrote claims (40 rows, 5 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds


## ステップ2 &mdash; 入れ子階層でソートする

`PROC NESTED` は、入力データが**CLASS変数の順に、最も外側の要因から先にソートされている**ことを要求する。`Region Branch Adjuster` の順にソートすることで、本プロシジャは階層を正しくたどることができる。

In [2]:
処理 並替 データ=claims;
   基準 Region Branch Adjuster;
実行;



NOTE: PROC SORT data=claims

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 40 rows from claims.
NOTE: Wrote claims (40 rows, 5 columns).
NOTE: PROC SORT statement used.


## ステップ3 &mdash; 保険金支払額の分散成分分析

中心となる分析。CLASS変数を最も外側から最も内側へ（`Region Branch Adjuster`）と列挙し、最も内側の反復である個々の請求は自動的に誤差項として扱われる。`VAR Settlement` は単一の応答変数を指定する。

`LABEL` ステートメントは、出力バナーにおける応答変数のより分かりやすい表示名を与える。出力には、期待平均平方の係数、分散分析表（各分散成分についてゼロに対するF検定を含む）、そして各レベルにおける**分散成分**の推定値とその**総分散に対する割合**が示される。

In [3]:
表題 '自動車保険金支払いの分散成分';
title2 '地域/支店/担当者のランダム効果ANOVA';

処理 nested データ=claims;
   分類 Region Branch Adjuster;
   変数 Settlement;
   見出 Region='地域' Branch='支店' Adjuster='担当者'
        Settlement = '支払保険金(USD)';
実行;


                                                     自動車保険金支払いの分散成分                                                     
                                                 地域/支店/担当者のランダム効果ANOVA                                                  


                       Nested Random Effects Analysis of Variance

Nested Random Effects Analysis of Variance for Variable 支払保険金(USD)
Variance Source       DF  Sum of Squares       F Value      Pr > F  Error Term   Mean Square  Variance Component    Percent of Total    EMS Coef
地域                     4  62114122.126866          6.71      0.0304        支店  15528530.531717  1651824.844989             54.4057      8.0000
支店                     5  11569658.859028          1.89      0.1829       担当者  2313931.771806   272682.828942              8.9813      4.0000
担当者                   10  12232004.560376          1.22      0.3348     Error  1223200.456038   111582.782346              3.6752      2.0000
Error                 20  20000697.826901         


NOTE: Option TITLE changed to 自動車保険金支払いの分散成分.
NOTE: Option TITLE2 changed to 地域/支店/担当者のランダム効果ANOVA.
NOTE: PROC NESTED nested ANOVA / variance components

NOTE: PROC NESTED statement used.


## ステップ4 &mdash; 2つの応答変数を同時に分析する

経営陣は**サイクルタイム**（請求の解決にかかる期間）も気にしている。`VAR` リストに `CycleDays` を追加する。従属変数が複数になると、`PROC NESTED` はさらに、階層の各レベルで2つの応答変数がどのように共変動するか（例えば、支払額が大きい地域は解決も遅いのか）を示す**共変動分析**を追加で報告する。

In [4]:
表題 '保険金支払いと解決期間の分散成分';
title2 '同一の入れ子階層における2つの応答変数';

処理 nested データ=claims;
   分類 Region Branch Adjuster;
   変数 Settlement CycleDays;
   見出 Region='地域' Branch='支店' Adjuster='担当者'
        Settlement = '支払保険金(USD)'
        CycleDays  = '解決日数';
実行;


                                                    保険金支払いと解決期間の分散成分                                                    
                                                  同一の入れ子階層における2つの応答変数                                                   


                       Nested Random Effects Analysis of Variance

Nested Random Effects Analysis of Variance for Variable 支払保険金(USD)
Variance Source       DF  Sum of Squares       F Value      Pr > F  Error Term   Mean Square  Variance Component    Percent of Total    EMS Coef
地域                     4  62114122.126866          6.71      0.0304        支店  15528530.531717  1651824.844989             54.4057      8.0000
支店                     5  11569658.859028          1.89      0.1829       担当者  2313931.771806   272682.828942              8.9813      4.0000
担当者                   10  12232004.560376          1.22      0.3348     Error  1223200.456038   111582.782346              3.6752      2.0000
Error                 20  20000697.826901         


NOTE: Option TITLE changed to 保険金支払いと解決期間の分散成分.
NOTE: Option TITLE2 changed to 同一の入れ子階層における2つの応答変数.
NOTE: PROC NESTED nested ANOVA / variance components

NOTE: PROC NESTED statement used.


## ステップ5 &mdash; AOV オプションによる分散成分のみの表示

両方の応答変数について分散成分の要約だけを手早く確認するために、`AOV` オプションは出力を分散分析の統計量に限定し、**共変動分析セクションを抑制する**。これは、応答変数ごとのレベル別分散内訳だけを必要とし、応答変数間の共変動までは不要なポートフォリオアナリストが目を通すのに適したコンパクトな表示である。

In [5]:
表題 '分散成分のみ(AOVオプション)';

処理 nested データ=claims aov;
   分類 Region Branch Adjuster;
   変数 Settlement CycleDays;
   見出 Region='地域' Branch='支店' Adjuster='担当者'
        Settlement = '支払保険金(USD)'
        CycleDays  = '解決日数';
実行;

表題;


                                                    分散成分のみ(AOVオプション)                                                    
                                                  同一の入れ子階層における2つの応答変数                                                   


                       Nested Random Effects Analysis of Variance

Nested Random Effects Analysis of Variance for Variable 支払保険金(USD)
Variance Source       DF  Sum of Squares       F Value      Pr > F  Error Term   Mean Square  Variance Component    Percent of Total    EMS Coef
地域                     4  62114122.126866          6.71      0.0304        支店  15528530.531717  1651824.844989             54.4057      8.0000
支店                     5  11569658.859028          1.89      0.1829       担当者  2313931.771806   272682.828942              8.9813      4.0000
担当者                   10  12232004.560376          1.22      0.3348     Error  1223200.456038   111582.782346              3.6752      2.0000
Error                 20  20000697.826901         


NOTE: Option TITLE changed to 分散成分のみ(AOVオプション).
NOTE: PROC NESTED nested ANOVA / variance components

NOTE: PROC NESTED statement used.


## 結果の解釈

分散分析表の**総分散に対する割合**列が本題である。上から下へ読むことで、総保険金支払いのばらつきのうち各組織階層に起因する割合が分かる。保険金支払額については、今回の実行で次の結果が得られた。

- **地域 — 54.4%**（Pr &gt; F = 0.0304）。データは地域レベルで最も大きなばらつきを持つよう生成されており、分析はそれを正しく復元している。総保険金支払いのばらつきの半分以上が地域*間*のものであり、請求が*どこで*処理されるかが支配的な要因であるという統計的に有意な証拠である。
- **支店（地域内） — 9.0%**（Pr &gt; F = 0.18）。地域から個々の支店まで掘り下げるとわずかに追加の割合が見られるが、ここでは有意ではない。
- **担当者（支店内） — 3.7%**（Pr &gt; F = 0.33）。この案件群では、個々の担当者に帰属できるばらつきはごくわずかである。
- **誤差 — 32.9%。** 担当者内の請求ごとの残差ノイズであり、これはどの組織的なレバーによっても取り除けない不可避のばらつきである。

各レベルには、その分散成分がゼロであるという帰無仮説に対する**F検定（Pr &gt; F）**も付随する。地域のp値が小さいこと（0.0304）は、標本抽出の偶然ではなく、地域間に真の系統的な差があることの統計的証拠である。

サイクルタイムの応答も同様のストーリーを示している。**地域が解決日数のばらつきの45.8%を占め**（Pr &gt; F = 0.0448、こちらも有意）、支店と担当者は一桁台の割合にとどまり、残差が40.1%を占める。したがって、保険会社が*いくら*支払うかも*どれだけの期間*かかるかも、第一に地域によって左右される。

2つの応答変数を同時に扱う実行では、**共変動分析**が追加される。ここでは地域レベルが積和を支配しており、全体の**相関係数は -0.36** — つまり負の関係である。支払額の大きい地域ほど、解決は*より速い*傾向にあり、遅いわけではない。これは有用で自明ではない発見である — 支払額の大きい地域が解決の遅い地域とは限らない。

**ビジネス上の示唆:** 地域が両方の応答変数について総分散に対する割合の内訳を支配しているため、保険会社はまず*地域をまたいで*精算ガイドラインと決裁権限を標準化すべきである — そこに最大かつ統計的に有意な不整合が存在するからだ — 支店レベルや担当者レベルの介入に投資するのはその後でよい。支払額とサイクルタイムの負の相関は、最も支払額が大きくかつ最も解決が遅い単一の地域は存在しないことを意味しており、この2つの課題は地域ごとに個別のプログラムで取り組むことができる。PROC NESTED は、「うちの精算は一貫性がない」という漠然とした感覚を、不整合がどこに存在するかを階層ごとに明らかにする実用的な帰属分析へと変える。